In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))
from seq2seq import *
import pandas as pd
import numpy as np
from multimodaltransformer_hyp import hyperparametersselection
from sklearn.model_selection import train_test_split 
from sklearn.utils import shuffle
from collections import Counter
import pickle
import torch
from sklearn.preprocessing import StandardScaler

In [2]:
def set_seeds(seed):
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
# Convert a string that simulates a list to a real list
def convert_string_list(element):
    # Delete [] of the string
    element = element[0:len(element)]
    # Create a list that contains each code as e.g. 'A'
    ATC_list = list(element.split('; '))
    for index, code in enumerate(ATC_list):
        # Delete '' of the code
        ATC_list[index] = code[0:len(code)]
    return ATC_list

In [4]:
def multiplicate_rows(df):
    # Duplicate each compound the number of ATC codes associated to it, copying its SMILES in new rows
    new_rows = []
    
    for _, row in df.iterrows():
        atc_codes = row['ATC Codes']
        atc_codes_list = convert_string_list(atc_codes)
        
        if len(atc_codes_list) > 1:
            for code in atc_codes_list:
                if len(code) == 5:
                    new_row = row.copy()
                    new_row['ATC Codes'] = code
                    new_rows.append(new_row)
        else:
            if len(atc_codes_list[0]) == 5:
                new_rows.append(row)
    
    new_set = pd.DataFrame(new_rows)
    new_set = new_set.reset_index(drop=True)

    return new_set

def extract_descriptors(df):
    """
    Extract molecular descriptors from your dataset.
    You'll need to implement this based on your descriptor source.
    
    Returns FloatTensor of shape (n_molecules, descriptor_dimension)
    """
    descriptors = df.iloc[:, 2:-5].values
    # Convert to numpy for easier handling
    if isinstance(descriptors, torch.Tensor):
        desc_array = descriptors.numpy()
    else:
        desc_array = np.array(descriptors)
    
    # Replace infinite values with NaN first
    desc_array[np.isinf(desc_array)] = np.nan
    
    # Calculate median for each feature (column-wise)
    medians = np.nanmedian(desc_array, axis=0)
    
    # Replace NaN values with corresponding median
    for i in range(desc_array.shape[1]):
        mask = np.isnan(desc_array[:, i])
        desc_array[mask, i] = medians[i]
    return torch.tensor(desc_array, dtype=torch.float32)
    
# Create vocabularies
# Tokenize the data
def source(df):
    source = []
    for compound in df['Neutralized SMILES']:
        # A list containing each SMILES character separated
        source.append(list(compound))
    return source
def target(df):
    target = []
    for codes in df['ATC Codes']:  
        code = convert_string_list(codes) 
        # A list of lists, each one containing each ATC code character separated 
        for c in code:
            list_c = list(c)
            target.append(list_c)
    return target

In [5]:
seeds = [42, 123, 47899, 2025, 1, 20, 99, 1020, 345, 78] 
columns = [
    'Seed', 
    'Precision', 'Recall', 'F1',
    'Precision level 1', 'Precision level 2', 'Precision level 3', 'Precision level 4',
    'Recall level 1', 'Recall level 2', 'Recall level 3', 'Recall level 4',
    '#Compounds that have at least one match'
]
metrics_df = pd.DataFrame(columns=columns)

for seed in seeds:
    set_seeds(seed)

    train_set = pd.read_csv(f'../Datasets/Rep_train_set{seed}.csv')
    test_set = pd.read_csv(f'../Datasets/Rep_test_set{seed}.csv')
    val_set = pd.read_csv(f'../Datasets/Rep_val_set{seed}.csv')
    
    new_train_set = multiplicate_rows(train_set)
    new_val_set = multiplicate_rows(val_set)
    new_test_set = multiplicate_rows(test_set)
    
    train_descriptors = extract_descriptors(new_train_set)
    test_descriptors = extract_descriptors(new_test_set)
    test_descriptors2 = extract_descriptors(test_set)
    val_descriptors = extract_descriptors(new_val_set)
    val_descriptors2 = extract_descriptors(val_set)
    
    scaler = StandardScaler()
    train_descriptors = torch.tensor(scaler.fit_transform(train_descriptors.numpy()), dtype=torch.float32)
    val_descriptors = torch.tensor(scaler.transform(val_descriptors.numpy()), dtype=torch.float32)
    test_descriptors = torch.tensor(scaler.transform(test_descriptors.numpy()), dtype=torch.float32)
    val_descriptors2 = torch.tensor(scaler.transform(val_descriptors2.numpy()), dtype=torch.float32)
    test_descriptors2 = torch.tensor(scaler.transform(test_descriptors2.numpy()), dtype=torch.float32)
    
    source_train = source(new_train_set)
    source_test = source(new_test_set)
    # Test set without duplicated compounds
    source_test2 = source(test_set)
    source_val = source(new_val_set)
    # Val set without duplicated compounds
    source_val2 = source(val_set)
    
    target_train = target(new_train_set)
    target_test = target(new_test_set)
    target_val = target(new_val_set)
    
    # An Index object represents a mapping from the vocabulary to integers (indices) to feed into the models
    source_index = index.Index(source_train)
    target_index = index.Index(target_train)
    
    # Create tensors
    X_train = source_index.text2tensor(source_train)
    y_train = target_index.text2tensor(target_train)
    X_val = source_index.text2tensor(source_val)
    X_val2 = source_index.text2tensor(source_val2)
    y_val = target_index.text2tensor(target_val)     
    X_test = source_index.text2tensor(source_test)
    X_test2 = source_index.text2tensor(source_test2)
    y_test = target_index.text2tensor(target_test)

    if torch.cuda.is_available():
        X_train = X_train.to("cuda")
        y_train = y_train.to("cuda")
        train_descriptors = train_descriptors.to("cuda") 
        test_descriptors = test_descriptors.to("cuda")
        test_descriptors2 = test_descriptors2.to("cuda")
        val_descriptors = val_descriptors.to("cuda")
        val_descriptors2 = val_descriptors2.to("cuda")
        X_val = X_val.to("cuda")
        X_val2 = X_val2.to("cuda")
        y_val = y_val.to("cuda")
        X_test= X_test.to("cuda")
        y_test = y_test.to("cuda")
        X_test2 = X_test2.to("cuda")

    if os.path.exists(f"s_mmtransformer_results{seed}.csv"):
        best_hyperparameters = (pd.read_csv(f"s_mmtransformer_results{seed}.csv")).loc[0]
    else:
        best_hyperparameters = hyperparametersselection(seed, source_index, target_index, X_train, X_val, X_val2, train_descriptors, val_descriptors, val_descriptors2, y_train, y_val)

    model = multimodal_models.MultimodalTransformer(
                source_index, 
                target_index,
                max_sequence_length = 800,
                embedding_dimension = best_hyperparameters['embedding_dim'],
                descriptors_dimension=train_descriptors.shape[1],
                feedforward_dimension = best_hyperparameters['feedforward_dim'],
                encoder_layers = best_hyperparameters['enc_layers'],
                decoder_layers = best_hyperparameters['dec_layers'],
                attention_heads = best_hyperparameters['attention_heads'],
                activation = "relu",
                dropout = best_hyperparameters['dropout'])   
    model.to("cuda")
    q = model.fit(X_train, 
            train_descriptors,
            y_train,
            X_val, 
            val_descriptors,
            y_val, 
            batch_size = 32, 
            epochs = 150, 
            learning_rate = best_hyperparameters['learning_rate'], 
            weight_decay = best_hyperparameters['weight_decay'],
            progress_bar = 0, 
            save_path = None)
    model.load_state_dict(torch.load("best_multimodalmodel.pth", weights_only=True))
    loss, error_rate = model.evaluate(X_test, test_descriptors, y_test, batch_size = 32) 

    predictions, log_probabilities = search_algorithms.multimodal_beam_search(
        model, 
        X_test2, # Make predictions with test set 
        test_descriptors2,
        predictions = 6, # max length of the predicted sequence
        beam_width = 10,
        batch_size = 32, 
        progress_bar = 0
    )
    output_beam = [target_index.tensor2text(p) for p in predictions]

    predictions_clean = []
    for i, preds in enumerate(output_beam):
        smiles = test_set.loc[i, 'Neutralized SMILES']
        interm = []
        for pred in preds:
            clean_pred = pred.replace('<START>', '').replace('<END>', '')
            if len(clean_pred) == 5:
                comp_in_train  = train_set.loc[train_set['Neutralized SMILES'] == smiles, 'ATC Codes']
                if comp_in_train.empty:
                    comp_in_val = val_set.loc[val_set['Neutralized SMILES'] == smiles, 'ATC Codes']
                    if clean_pred not in convert_string_list(val_set.loc[val_set['Neutralized SMILES'] == smiles, 'ATC Codes'].iloc[0]):
                        interm.append(clean_pred)
                else:
                    if clean_pred not in convert_string_list(train_set.loc[train_set['Neutralized SMILES'] == smiles, 'ATC Codes'].iloc[0]):
                        interm.append(clean_pred)
            if len(interm) == 3:
                break
        predictions_clean.append(interm)
    precision_1, precision_2, precision_3, precision_4 = defined_metrics.precision(predictions_clean, f'../Datasets/Rep_test_set{seed}.csv', 'ATC Codes')
    recall_1, recall_2, recall_3, recall_4, counter_compound_match = defined_metrics.recall(predictions_clean, f'../Datasets/Rep_test_set{seed}.csv', 'ATC Codes')
    precisions, recalls, f1s = defined_metrics.complete_metrics(predictions_clean, f'../Datasets/Rep_test_set{seed}.csv', 'ATC Codes', 3)
    precisions_average = sum(precisions)/len(precisions)
    recalls_average = sum(recalls)/len(recalls)
    f1s_average = sum(f1s)/len(f1s)
    
    metrics = {
        'Precision': precisions_average, 
        'Recall': recalls_average,
        'F1': f1s_average,
        'Precision level 1': precision_1,
        'Precision level 2': precision_2,
        'Precision level 3': precision_3,
        'Precision level 4': precision_4,
        'Recall level 1': recall_1,
        'Recall level 2': recall_2,
        'Recall level 3': recall_3,
        'Recall level 4': recall_4,
        '#Compounds that have at least one match': counter_compound_match
    }
    
    # Build the row
    row = {
        'Seed': seed,
        **metrics
    }
    
    metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)

    torch.cuda.empty_cache()

metrics_df.to_csv("multimodaltransformer_metrics.csv", index=False)
print("Mean:", metrics_df.mean(numeric_only=True))
print("Std:", metrics_df.std(numeric_only=True))

C:\Users\trini\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Model: Seq2Seq Multimodal Transformer
Source index: <Seq2Seq Index with 46 items>
Target index: <Seq2Seq Index with 34 items>
Max sequence length: 800
Embedding dimension: 128
Descriptors dimension: 1136
Feedforward dimension: 256
Encoder layers: 3
Decoder layers: 4
Attention heads: 4
Activation: relu
Dropout: 0.0
Trainable parameters: 1,472,034

Training started
X_train.shape: torch.Size([3024, 702])
Y_train.shape: torch.Size([3024, 7])
X_dev.shape: torch.Size([538, 295])
Y_dev.shape: torch.Size([538, 7])
Epochs: 150
Learning rate: 0.0001
Weight decay: 1e-05
Epoch | Train                 | Development           | Minutes
      | Loss     | Error Rate | Loss     | Error Rate |
---------------------------------------------------------------
    1 |   1.9841 |     51.924 |   1.4033 |     42.565 |     0.2
    2 |   1.2968 |     39.076 |   1.2442 |     38.724 |     0.3
    3 |   1.1596 |     35.676 |   1.1384 |     36.152 |     0.5
    4 |   1.0551 |     32.788 |   1.0646 |     34.108 |   

C:\Users\trini\AppData\Local\Temp\ipykernel_139488\690283428.py:166: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)
C:\Users\trini\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Model: Seq2Seq Multimodal Transformer
Source index: <Seq2Seq Index with 45 items>
Target index: <Seq2Seq Index with 34 items>
Max sequence length: 800
Embedding dimension: 64
Descriptors dimension: 1136
Feedforward dimension: 128
Encoder layers: 3
Decoder layers: 3
Attention heads: 4
Activation: relu
Dropout: 0.0
Trainable parameters: 386,722

Training started
X_train.shape: torch.Size([3028, 649])
Y_train.shape: torch.Size([3028, 7])
X_dev.shape: torch.Size([534, 702])
Y_dev.shape: torch.Size([534, 7])
Epochs: 150
Learning rate: 0.001
Weight decay: 0.0001
Epoch | Train                 | Development           | Minutes
      | Loss     | Error Rate | Loss     | Error Rate |
---------------------------------------------------------------
    1 |   1.6496 |     46.967 |   1.2602 |     40.169 |     0.1
    2 |   1.1907 |     38.926 |   1.1216 |     37.672 |     0.2
    3 |   1.0590 |     35.227 |   1.0502 |     34.145 |     0.3
    4 |   0.9860 |     33.020 |   0.9599 |     31.742 |     0

C:\Users\trini\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


...

Model: Seq2Seq Multimodal Transformer
Source index: <Seq2Seq Index with 46 items>
Target index: <Seq2Seq Index with 33 items>
Max sequence length: 800
Embedding dimension: 128
Descriptors dimension: 1136
Feedforward dimension: 128
Encoder layers: 3
Decoder layers: 4
Attention heads: 2
Activation: relu
Dropout: 0.0
Trainable parameters: 1,241,505

Training started
X_train.shape: torch.Size([3025, 702])
Y_train.shape: torch.Size([3025, 7])
X_dev.shape: torch.Size([537, 337])
Y_dev.shape: torch.Size([537, 7])
Epochs: 150
Learning rate: 0.0001
Weight decay: 1e-05
Epoch | Train                 | Development           | Minutes
      | Loss     | Error Rate | Loss     | Error Rate |
---------------------------------------------------------------
    1 |   2.1367 |     55.697 |   1.4970 |     44.072 |     0.1
    2 |   1.3689 |     41.196 |   1.3056 |     40.751 |     0.2
    3 |   1.2191 |     37.620 |   1.2009 |     37.834 |     0.4
    4 |   1.1110 |     34.430 |   1.0885 |     33.768 |   

C:\Users\trini\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Model: Seq2Seq Multimodal Transformer
Source index: <Seq2Seq Index with 45 items>
Target index: <Seq2Seq Index with 33 items>
Max sequence length: 800
Embedding dimension: 128
Descriptors dimension: 1136
Feedforward dimension: 256
Encoder layers: 4
Decoder layers: 4
Attention heads: 2
Activation: relu
Dropout: 0.0
Trainable parameters: 1,604,129

Training started
X_train.shape: torch.Size([3033, 702])
Y_train.shape: torch.Size([3033, 7])
X_dev.shape: torch.Size([529, 267])
Y_dev.shape: torch.Size([529, 7])
Epochs: 150
Learning rate: 0.001
Weight decay: 0.0001
Epoch | Train                 | Development           | Minutes
      | Loss     | Error Rate | Loss     | Error Rate |
---------------------------------------------------------------
    1 |   1.5132 |     47.181 |   1.2650 |     41.241 |     0.2
    2 |   1.1895 |     40.340 |   1.1226 |     38.248 |     0.3
    3 |   1.0866 |     36.911 |   1.0974 |     36.578 |     0.5
    4 |   1.0195 |     34.910 |   1.0344 |     34.846 |   

C:\Users\trini\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Model: Seq2Seq Multimodal Transformer
Source index: <Seq2Seq Index with 46 items>
Target index: <Seq2Seq Index with 34 items>
Max sequence length: 800
Embedding dimension: 128
Descriptors dimension: 1136
Feedforward dimension: 256
Encoder layers: 3
Decoder layers: 4
Attention heads: 4
Activation: relu
Dropout: 0.0
Trainable parameters: 1,472,034

Training started
X_train.shape: torch.Size([3017, 702])
Y_train.shape: torch.Size([3017, 7])
X_dev.shape: torch.Size([545, 323])
Y_dev.shape: torch.Size([545, 7])
Epochs: 150
Learning rate: 0.0001
Weight decay: 0.0001
Epoch | Train                 | Development           | Minutes
      | Loss     | Error Rate | Loss     | Error Rate |
---------------------------------------------------------------
    1 |   1.9493 |     50.691 |   1.4128 |     42.661 |     0.1
    2 |   1.2939 |     39.559 |   1.2580 |     38.226 |     0.3
    3 |   1.1480 |     35.620 |   1.1819 |     35.933 |     0.5
    4 |   1.0478 |     32.516 |   1.1436 |     33.884 |  